### Ricostruzione Compagine Sociale dalla Stampa

Questo notebook implementa una pipeline end-to-end per estrarre eventi sulla compagine sociale
(round di finanziamento, cessioni di quote, ecc.) di startup italiane a partire da articoli
di stampa online.

### Flusso
```
Aziende 
    ↓
[Step 1] Ricerca articoli con Tavily 
    ↓
[Step 2] Estrazione strutturata con OpenAI GPT-4o-mini 
    ↓
[Step 3] Verifica anti-allucinazione (fuzzy matching)
    ↓
[Step 4] Output → Excel + JSON
```


## 1. Import e logging

In [33]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [1]:
import argparse
import csv
import json
import logging
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from openai import OpenAI
from rapidfuzz import fuzz

# Logging leggibile in notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


## 2. Configurazione

Impostiamo le API key e i parametri principali.


In [2]:
# ── API keys ────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY",
    "chiave openAI"
)
TAVILY_API_KEY = os.getenv(
    "TAVILY_API_KEY",
    "chiave_tavily"
)

# ── Modello e path ───────────────────────────────────────────────────────────
MODEL        = "gpt-4o-mini"
OUTPUT       = Path("output")
DEFAULT_INPUT = Path("Analisi-AIDA.xlsx")

# ── Template di ricerca Tavily ───────────────────────────────────────────────
QUERY_TEMPLATE = [
    '"{nome}" round investimento raccolta milioni',      # eventi primari
    '"{nome}" cessione quote soci secondario liquidita', # eventi secondari
]

# ── Parametri di esecuzione ──────────────────────────────────────────────────
MAX_TAVILY_WORKERS       = 4   # richieste Tavily in parallelo
MAX_OPENAI_WORKERS       = 4   # chiamate OpenAI in parallelo
MAX_RISULTATI_PER_QUERY  = 1   # 1 risultato × 2 query = 2 articoli per azienda
MIN_TESTO_LEN            = 500    # scarta articoli troppo corti
MAX_TESTO_LEN            = 25_000 # tronca prima di mandare al modello
API_MAX_RETRIES          = 1
API_RETRY_DELAY          = 2.0    # secondi tra tentativi
DEFAULT_BATCH_SIZE       = 50

RESUME = True
INPUT_FILE = DEFAULT_INPUT

print(" Configurazione caricata")
print(f"   Modello     : {MODEL}")
print(f"   Output dir  : {OUTPUT}")
print(f"   OpenAI key  : {'ok' if OPENAI_API_KEY else 'MANCA ⚠️'}")
print(f"   Tavily key  : {'ok' if TAVILY_API_KEY else 'MANCA ⚠️'}")


 Configurazione caricata
   Modello     : gpt-4o-mini
   Output dir  : output
   OpenAI key  : ok
   Tavily key  : ok


## 3. Lettura dati aziende e pulizia nomi

Le ragioni sociali nei file AIDA contengono suffissi legali ("S.R.L.", "S.P.A.", ecc.)
che potrebbero peggiorare la qualità delle query di ricerca --> le togliamo


In [3]:
# Pattern legali da rimuovere per ottenere un nome pulito per le query
_LEGAL_SUFFIXES = re.compile(
    r"""
    (?:
        \(.*?(?:START[\s-]*UP|COSTITUITA|DECRETO|ART\.|LEGGE|COMMA).*?\)  # testo legale in parentesi
    )
    |
    \b(?:
        S\.?R\.?L\.?(?:\s*SEMPLIFICATA)?|
        S\.?P\.?A\.?|S\.?A\.?S\.?|S\.?N\.?C\.?|S\.?B\.?|
        S\.?C\.?A\.?R\.?L\.?|
        SOCIETA['\s]*A\s*RESPONSABILITA['\s]*LIMITATA|
        SOCIETA['\s]*PER\s*AZIONI|
        START[\s-]*UP(?:\s+INNOVATIVA)?|
        SOCIETA['\s]*BENEFIT|
        IN\s+FORMA\s+ABBREVIATA\s+\S+|
        ABBREVIABILE\s+ANCHE\s+COME\s+.*
    )\b
    """,
    re.IGNORECASE | re.VERBOSE,
)


def pulisci_nome(ragione_sociale: str) -> str:
    """estrae il nome 'commerciale' dalla ragione sociale completa.

    Esempi:
        'SIKELIA OIL S.R.L.'  →  'Sikelia Oil'
        'JET HR S.R.L.'       →  'Jet Hr'
        "B2CONNECT SOCIETA A RESPONSABILITA' LIMITATA ..." → 'B2connect'
    """
    nome = _LEGAL_SUFFIXES.sub("", ragione_sociale)
    nome = re.sub(r"\(.*(?:COSTITUITA|DECRETO|ART\.|LEGGE|COMMA).*$", "", nome, flags=re.IGNORECASE)
    nome = re.sub(r"[^\w\s]", " ", nome)
    nome = re.sub(r"\s+", " ", nome).strip()
    parti = []
    for p in nome.split():
        if len(p) <= 3 and p.isupper():
            parti.append(p)
        else:
            parti.append(p.capitalize())
    return " ".join(parti)


# Test rapido
for esempio in ["SIKELIA OIL S.R.L.", "JET HR S.R.L.", "B2CONNECT SOCIETA A RESPONSABILITA' LIMITATA"]:
    print(f"{esempio!r:55s} → {pulisci_nome(esempio)!r}")


'SIKELIA OIL S.R.L.'                                    → 'Sikelia OIL'
'JET HR S.R.L.'                                         → 'JET HR'
"B2CONNECT SOCIETA A RESPONSABILITA' LIMITATA"          → 'B2connect'


### 3.1 Caricamento del file aziende (CSV / XLSX)


In [4]:
def _find_column(columns: list[str], keyword: str) -> str | None:
    """Trova una colonna il cui nome contiene `keyword` (case-insensitive)."""
    kw = keyword.lower()
    for c in columns:
        if kw in str(c).lower().replace("\n", " "):
            return c
    return None


def load_companies(file_path: Path) -> list[dict]:
    """Leggi un file CSV (;) o XLSX e restituisci una lista di dizionari."""
    ext = file_path.suffix.lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(file_path, dtype=str).fillna("")
    elif ext == ".csv":
        df = pd.read_csv(file_path, sep=";", encoding="utf-8-sig", dtype=str).fillna("")
    else:
        raise ValueError(f"Formato non supportato: {ext}  (usa .csv o .xlsx)")

    col_rs   = _find_column(df.columns.tolist(), "ragione sociale")
    col_cf   = _find_column(df.columns.tolist(), "codice fiscale")
    col_web  = _find_column(df.columns.tolist(), "website")
    col_atc  = _find_column(df.columns.tolist(), "ateco 2007")
    col_stat = _find_column(df.columns.tolist(), "stato giuridico")

    if col_rs is None:
        raise ValueError("Colonna 'Ragione sociale' non trovata nel file")

    aziende = []
    for _, row in df.iterrows():
        ragione_sociale = str(row.get(col_rs, "")).strip()
        if not ragione_sociale:
            continue
        nome_pulito = pulisci_nome(ragione_sociale)
        if not nome_pulito or len(nome_pulito) < 2:
            continue
        aziende.append({
            "nome":           nome_pulito,
            "ragione_sociale": ragione_sociale,
            "codice_fiscale": str(row.get(col_cf, "")) if col_cf else "",
            "website":        str(row.get(col_web, "")) if col_web else "",
            "ateco_codice":   str(row.get(col_atc, "")) if col_atc else "",
            "stato_giuridico": str(row.get(col_stat, "")) if col_stat else "",
        })
    log.info("Caricate %d aziende da %s", len(aziende), file_path)
    return aziende


def load_already_processed_names(output_dir: Path) -> set[str]:
    """Leggi i nomi già presenti nell'output JSON per il resume."""
    json_path = output_dir / "eventi_estratti.json"
    if not json_path.exists():
        return set()
    try:
        data = json.loads(json_path.read_text(encoding="utf-8"))
        return {ev.get("azienda", "") for ev in data}
    except Exception:
        return set()


print("Funzioni di caricamento definite")


Funzioni di caricamento definite


### 3.2 Seleziona il file e il batch da processare

Modifichiamo le variabili in questa cella per cambiare input, range o modalità resume.


In [5]:
# ── Parametri di esecuzione ──────────────────────────────────────────────────
START      = 0
BATCH_SIZE = 100
RESUME     = True

# ── Funzione progresso ────────────────────────────────────────────────────────
def load_progress(output_dir: Path) -> set[str]:
    progress_path = output_dir / "progresso.json"
    if not progress_path.exists():
        return set()
    try:
        return set(json.loads(progress_path.read_text(encoding="utf-8")))
    except Exception:
        return set()

# ── Caricamento ───────────────────────────────────────────────────────────────
tutte_aziende = load_companies(INPUT_FILE)

if RESUME:
    gia_fatte = load_progress(OUTPUT)
    tutte_aziende = [az for az in tutte_aziende if az["nome"] not in gia_fatte]
    log.info("Resume: %d rimanenti", len(tutte_aziende))

batch = tutte_aziende[START : START + BATCH_SIZE]
log.info("Batch: %d aziende (indici %d-%d)", len(batch), START, START + len(batch) - 1)

# Anteprima
pd.DataFrame(batch).head(10)


18:43:12 [INFO] Caricate 715 aziende da Analisi-AIDA.xlsx
18:43:12 [INFO] Resume: 715 rimanenti
18:43:12 [INFO] Batch: 100 aziende (indici 0-99)


,nome,ragione_sociale,codice_fiscale,website,ateco_codice,stato_giuridico
0,Prolux,PROLUX S.R.L,2325970685,www.progettarredo.it,310000,Attiva
1,Original Brands,ORIGINAL BRANDS S.R.L.,12980680966,www.originalbrands.group,631130,Attiva
2,Subbyx,SUBBYX S.R.L. - SOCIETA' BENEFIT,12913000969,www.subbyx.com,620100,Attiva
3,Metalmed,METALMED SRL,3031620309,,325020,Attiva
4,Officine Emons,OFFICINE EMONS S.R.L.,16749701005,,592010,Attiva
5,Feat Venture Studio,FEAT VENTURE STUDIO S.R.L. S.B. ABBREVIABILE A...,12578030012,featventures.com,620100,Attiva
6,Ntry,NTRY S.R.L.,17556621005,www.ntry.it,620100,Attiva
7,Consolidata,CONSOLIDATA SRL,12118260962,www.consolidata.tech,620100,Attiva
8,Ricreatec,RICREATEC S.R.L.,1667810558,www.ricreatec.it,289999,Attiva
9,Smart AT Work,SMART AT WORK S.R.L.,13235270967,www.smartatwork.it,620200,Attiva


## 4. Prompt di estrazione e schema JSON

Il modello riceve il testo di ogni articolo e deve estrarre eventi strutturati.


In [6]:
EXTRACTION_PROMPT = """\
You are a financial analyst extracting structured data from Italian business press articles.

FUNDAMENTAL RULE — read this before anything else:
Extract ONLY information explicitly written in the article provided. Never use your prior knowledge
about this company, its investors, or its funding rounds. If a piece of information is not in the
text, the correct value is `null` (or an empty list). A plausible but unwritten value is a serious error.

DEFINITIONS
- PRIMARY: new money enters the company (capital increase, new shares issued).
- SECONDARY: existing shares change hands; the money goes to the SELLING shareholders, not to the
  company. Typical signals in Italian: "cessione di quote", "i soci storici hanno monetizzato",
  "componente secondaria", "shareholder liquidity", "disinvestimento".
- MIXED: the deal contains both components (common in late-stage rounds — pay close attention).
- If the text does not allow you to tell: use "undetermined". Do not guess.

FIELD RULES
- Convert amounts to euros as integers ("12 milioni" -> 12000000); keep the original wording in `amount.text`.
- Dates: use the event date if stated, otherwise the article publication date, with the actual granularity.
- For every event, `evidence` must contain at least one quote COPIED VERBATIM from the article
  (max 40 words). Do not paraphrase: quotes are automatically checked against the original text,
  and an event whose quote is not found is discarded.
- `confidence`: 0.9+ if stated literally; 0.5-0.7 if inferred from indirect wording; below 0.5 if very uncertain.
- If the article describes no shareholder-structure event, return `events: []`.

I repeat the fundamental rule: only what is written in the article. Nothing else.\
"""

SCHEMA = {
    "name": "shareholder_event_extraction",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "required": ["company_match", "events"],
        "properties": {
            "company_match": {
                "type": "boolean",
                "description": "True if the article is really about the target company",
            },
            "events": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": [
                        "event_type", "round_type", "date", "amount",
                        "investors_in", "shareholders_out", "evidence", "confidence",
                    ],
                    "properties": {
                        "event_type": {
                            "type": "string",
                            "enum": ["funding_round", "capital_increase", "share_sale",
                                     "full_exit", "buyback", "other"],
                        },
                        "round_type": {
                            "type": "string",
                            "enum": ["primary", "secondary", "mixed", "undetermined"],
                        },
                        "date": {
                            "type": ["string", "null"],
                            "description": "ISO 8601, partial allowed: YYYY, YYYY-MM or YYYY-MM-DD",
                        },
                        "amount": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": ["eur", "text"],
                            "properties": {
                                "eur":  {"type": ["number", "null"]},
                                "text": {"type": ["string", "null"]},
                            },
                        },
                        "investors_in": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "required": ["name", "lead"],
                                "properties": {
                                    "name": {"type": "string"},
                                    "lead": {"type": ["boolean", "null"]},
                                },
                            },
                        },
                        "shareholders_out": {
                            "type": "array",
                            "description": "Shareholders who sold or liquidated stakes",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "required": ["name", "stake_sold_pct"],
                                "properties": {
                                    "name":           {"type": "string"},
                                    "stake_sold_pct": {"type": ["number", "null"]},
                                },
                            },
                        },
                        "evidence": {
                            "type": "array",
                            "minItems": 1,
                            "items": {
                                "type": "string",
                                "description": "verbatim quote from the article, max 40 words",
                            },
                        },
                        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                    },
                },
            },
        },
    },
}

print("Prompt e schema definiti")


Prompt e schema definiti


## Step 1 — Ricerca articoli con Tavily

Per ogni azienda vengono lanciate 2 query in parallelo (eventi primari e secondari).
I risultati vengono deduplicati per URL e filtrati per lunghezza minima.


In [7]:
def _tavily_query(nome: str, template: str, max_risultati: int) -> list[dict]:
    """Singola query Tavily con retry."""
    query = template.format(nome=nome)
    for attempt in range(1, API_MAX_RETRIES + 1):
        try:
            resp = requests.post(
                "https://api.tavily.com/search",
                json={
                    "api_key":           TAVILY_API_KEY,
                    "query":             query,
                    "max_results":       max_risultati,
                    "search_depth":      "advanced",
                    "include_raw_content": True,
                },
                timeout=30,
            )
            resp.raise_for_status()
            return resp.json().get("results", [])
        except Exception as exc:
            if attempt < API_MAX_RETRIES:
                log.warning("Tavily tentativo %d/%d fallito per '%s': %s", attempt, API_MAX_RETRIES, nome, exc)
                time.sleep(API_RETRY_DELAY)
            else:
                log.error("Tavily abbandonato per '%s' dopo %d tentativi: %s", nome, API_MAX_RETRIES, exc)
                return []


def search_articles_parallel(aziende: list[dict]) -> list[dict]:
    """Cerca articoli per tutte le aziende in parallelo (ThreadPoolExecutor)."""
    tasks = [
        (az["nome"], tmpl)
        for az in aziende
        for tmpl in QUERY_TEMPLATE
    ]

    raw_results: dict[tuple, list] = {}
    with ThreadPoolExecutor(max_workers=MAX_TAVILY_WORKERS) as pool:
        future_to_key = {
            pool.submit(_tavily_query, nome, tmpl, MAX_RISULTATI_PER_QUERY): (nome, tmpl)
            for nome, tmpl in tasks
        }
        for future in as_completed(future_to_key):
            key = future_to_key[future]
            raw_results[key] = future.result()

    # Assembla e deduplica per URL
    corpus: list[dict] = []
    visti: set[str] = set()
    for az in aziende:
        nome = az["nome"]
        for tmpl in QUERY_TEMPLATE:
            for r in raw_results.get((nome, tmpl), []):
                url   = (r.get("url") or "").split("?")[0].rstrip("/")
                testo = r.get("raw_content") or r.get("content") or ""
                if url in visti or len(testo) < MIN_TESTO_LEN:
                    continue
                visti.add(url)
                corpus.append({
                    "azienda":         nome,
                    "ragione_sociale": az.get("ragione_sociale", ""),
                    "codice_fiscale":  az.get("codice_fiscale", ""),
                    "url":             r.get("url"),
                    "titolo":          r.get("title"),
                    "data":            r.get("published_date"),
                    "testo":           testo,
                })
        n_az = sum(1 for a in corpus if a["azienda"] == nome)
        log.info("%-25s articoli utilizzabili: %d", nome, n_az)

    log.info("Corpus totale: %d articoli", len(corpus))
    return corpus


print("Funzioni Tavily definite")


Funzioni Tavily definite


In [ ]:
# ▶ Esegui la ricerca articoli
DRY_RUN = False

if not DRY_RUN and batch:
    log.info("--- Step 1: ricerca articoli ---")
    corpus = search_articles_parallel(batch)
    print(f"\n📰 Articoli trovati: {len(corpus)}")
    if corpus:
        pd.DataFrame(corpus)[['azienda','titolo','data','url']].head(10)
else:
    corpus = []
    print("DRY_RUN attivo — step saltato")

09:57:58 [INFO] --- Step 1: ricerca articoli ---


09:58:19 [ERROR] Tavily abbandonato per 'Reskilla' dopo 1 tentativi: 432 Client Error:  for url: https://api.tavily.com/search
09:58:19 [ERROR] Tavily abbandonato per 'Domus Group' dopo 1 tentativi: 432 Client Error:  for url: https://api.tavily.com/search
09:58:19 [ERROR] Tavily abbandonato per 'Domus Group' dopo 1 tentativi: 432 Client Error:  for url: https://api.tavily.com/search
09:58:19 [ERROR] Tavily abbandonato per 'Nigesa' dopo 1 tentativi: 432 Client Error:  for url: https://api.tavily.com/search
09:58:20 [ERROR] Tavily abbandonato per 'Nigesa' dopo 1 tentativi: 432 Client Error:  for url: https://api.tavily.com/search
09:58:20 [ERROR] Tavily abbandonato per 'Smace' dopo 1 tentativi: 432 Client Error:  for url: https://api.tavily.com/search
09:58:20 [ERROR] Tavily abbandonato per 'Mama Innovation' dopo 1 tentativi: 432 Client Error:  for url: https://api.tavily.com/search
09:58:20 [ERROR] Tavily abbandonato per 'Smace' dopo 1 tentativi: 432 Client Error:  for url: https://api


📰 Articoli trovati: 83


## Step 2 — Estrazione strutturata con OpenAI

Ogni articolo viene inviato al modello con il prompt di sistema e lo schema JSON.


In [8]:
def _extract_article(client: OpenAI, articolo: dict) -> dict | None:
    """Chiamata OpenAI con retry; restituisce None in caso di errore persistente."""
    user_msg = (
        f"TARGET COMPANY: {articolo['azienda']}\n\n"
        f"ARTICLE\nTitle: {articolo['titolo']}\n"
        f"Published: {articolo['data'] or 'unknown'}\n\n"
        f"{articolo['testo'][:MAX_TESTO_LEN]}"
    )
    for attempt in range(1, API_MAX_RETRIES + 1):
        try:
            risposta = client.chat.completions.create(
                model=MODEL,
                temperature=0,
                messages=[
                    {"role": "system", "content": EXTRACTION_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                response_format={"type": "json_schema", "json_schema": SCHEMA},
            )
            return json.loads(risposta.choices[0].message.content)
        except Exception as exc:
            if attempt < API_MAX_RETRIES:
                log.warning("OpenAI tentativo %d/%d fallito: %s", attempt, API_MAX_RETRIES, exc)
                time.sleep(API_RETRY_DELAY)
            else:
                log.error("OpenAI abbandonato per '%s': %s", articolo["titolo"][:60], exc)
                return None


def extract_corpus_parallel(corpus: list[dict]) -> list[dict]:
    """Lancia l'estrazione OpenAI su tutto il corpus in parallelo."""
    client = OpenAI(api_key=OPENAI_API_KEY)
    risultati: list[dict] = []

    with ThreadPoolExecutor(max_workers=MAX_OPENAI_WORKERS) as pool:
        future_to_art = {
            pool.submit(_extract_article, client, art): art
            for art in corpus
        }
        for future in as_completed(future_to_art):
            articolo = future_to_art[future]
            out = future.result()
            if out is None:
                continue
            if not out["company_match"]:
                log.info("  scartato (omonimia): %s", articolo["titolo"][:60])
                continue
            for evento in out["events"]:
                evento["azienda"]          = articolo["azienda"]
                evento["ragione_sociale"]  = articolo.get("ragione_sociale", "")
                evento["codice_fiscale"]   = articolo.get("codice_fiscale", "")
                evento["fonte_url"]        = articolo["url"]
                evento["fonte_data"]       = articolo["data"]
                evento["fonte_testo"]      = articolo["testo"]
                risultati.append(evento)
            log.info("%-25s %d eventi da: %s", articolo["azienda"], len(out["events"]), articolo["titolo"][:55])

    log.info("Eventi estratti in totale: %d", len(risultati))
    return risultati


print("Funzioni OpenAI definite")


Funzioni OpenAI definite


In [ ]:
# ▶ Esegui l'estrazione LLM
if corpus:
    log.info("--- Step 2: estrazione LLM ---")
    risultati = extract_corpus_parallel(corpus)
    print(f"\n🔍 Eventi estratti: {len(risultati)}")
else:
    risultati = []
    print("Corpus vuoto — step saltato")


09:58:32 [INFO] --- Step 2: estrazione LLM ---
09:58:33 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:58:33 [INFO]   scartato (omonimia): Startup, i round e le operazioni di ottobre 2025 - la Repubb
09:58:33 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:58:33 [INFO]   scartato (omonimia): Cessione quote SRL: tassazione, costi e come risparmiare
09:58:33 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:58:33 [INFO]   scartato (omonimia): CESSIONE QUOTE SRL, COME OTTIMIZZARE L'IMPATTO FISCALE.
09:58:34 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:58:34 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:58:34 [INFO]   scartato (omonimia): Cessione di quote societarie dal Notaio
09:58:34 [INFO]   scartato (omonimia): Recesso del socio o cessione quote? Quale conviene?
09:58:34 [IN


🔍 Eventi estratti: 20


## Step 3 — Verifica anti-allucinazione

Ogni evento viene confrontato con il testo originale dell'articolo tramite fuzzy matching
(libreria `rapidfuzz`). Gli eventi vengono classificati in:

-  verificato >>>  Tutte le citazioni e i nomi trovati nel testo 
- sospetto >>> Alcune citazioni o nomi non trovati 
- scartato >>> Nessuna citazione trovata nel testo 


In [9]:
def normalize(t: str) -> str:
    """Normalizza il testo per il confronto fuzzy."""
    return re.sub(r"\s+", " ", str(t)).replace("'", "'").replace("\u201c", '"').replace("\u201d", '"').lower()


def verify_events(risultati: list[dict]) -> list[dict]:
    """Marca ogni evento come verificato / sospetto / scartato."""
    for ev in risultati:
        testo = normalize(ev["fonte_testo"])
        citazioni_ok = sum(
            fuzz.partial_ratio(normalize(q), testo) >= 90
            for q in ev["evidence"]
        )
        nomi = [i["name"] for i in ev["investors_in"]] + [s["name"] for s in ev["shareholders_out"]]
        nomi_fantasma = [n for n in nomi if fuzz.partial_ratio(normalize(n), testo) < 90]

        if citazioni_ok == 0:
            ev["verifica"] = "scartato"
        elif nomi_fantasma or citazioni_ok < len(ev["evidence"]):
            ev["verifica"] = "sospetto"
            ev["nomi_non_trovati"] = nomi_fantasma
        else:
            ev["verifica"] = "verificato"

    conteggio = pd.Series([ev["verifica"] for ev in risultati]).value_counts()
    log.info("Verifica:\n%s", conteggio.to_string())
    return risultati


print("Funzione di verifica definita")


Funzione di verifica definita


In [ ]:
# ▶ Esegui la verifica anti-allucinazione
if risultati:
    log.info("--- Step 3: verifica anti-allucinazione ---")
    risultati = verify_events(risultati)
    conteggio = pd.Series([ev["verifica"] for ev in risultati]).value_counts()
    print("\n📊 Riepilogo verifica:")
    display(conteggio.to_frame("conteggio"))
else:
    print("Nessun evento da verificare")


09:59:09 [INFO] --- Step 3: verifica anti-allucinazione ---
09:59:09 [INFO] Verifica:
verificato    14
sospetto       3
scartato       3



📊 Riepilogo verifica:


,conteggio
verificato,14
sospetto,3
scartato,3


## Step 4 — Salvataggio output

I risultati vengono salvati in:
- output/eventi_estratti.xlsx — tabella Excel
- output/eventi_estratti.json — JSON completo (senzùa il testo dell'articolo)
- output/progresso.json — lista aziende processate 


In [10]:
def save_output(risultati: list[dict], output_dir: Path, append: bool = False) -> None:
    """Salva i risultati. Se append=True, aggiunge ai file esistenti."""
    output_dir.mkdir(exist_ok=True)
    xlsx_path = output_dir / "eventi_estratti.xlsx"
    json_path = output_dir / "eventi_estratti.json"

    risultati_totali = risultati
    if append and json_path.exists():
        try:
            precedenti = json.loads(json_path.read_text(encoding="utf-8"))
            risultati_totali = precedenti + risultati
            log.info("Append: %d nuovi + %d precedenti = %d totali",
                     len(risultati), len(precedenti), len(risultati_totali))
        except Exception:
            log.warning("Impossibile leggere JSON precedente, sovrascrittura")

    tabella = pd.DataFrame([{
        "azienda":         ev["azienda"],
        "ragione_sociale": ev.get("ragione_sociale", ""),
        "codice_fiscale":  ev.get("codice_fiscale", ""),
        "data":            ev["date"],
        "tipo_evento":     ev["event_type"],
        "tipo_round":      ev["round_type"],
        "importo_eur":     ev["amount"]["eur"],
        "investitori_in":  "; ".join(i["name"] for i in ev["investors_in"]),
        "soci_uscenti":    "; ".join(s["name"] for s in ev["shareholders_out"]),
        "confidence":      ev["confidence"],
        "verifica":        ev["verifica"],
        "evidence":        " | ".join(ev["evidence"])[:300],
        "fonte_url":       ev["fonte_url"],
    } for ev in risultati_totali]).sort_values(["azienda", "data"])

    tabella.to_excel(xlsx_path, index=False)
    json_path.write_text(
        json.dumps(
            [{k: v for k, v in ev.items() if k != "fonte_testo"} for ev in risultati_totali],
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    log.info("Salvati: %s  e  %s  (%d eventi totali)", xlsx_path, json_path, len(risultati_totali))
    return tabella


def save_progress(aziende_processate: list[str], output_dir: Path) -> None:
    """Salva la lista di aziende già processate per il resume."""
    output_dir.mkdir(exist_ok=True)
    progress_path = output_dir / "progresso.json"
    nomi_set = set()
    if progress_path.exists():
        try:
            nomi_set = set(json.loads(progress_path.read_text(encoding="utf-8")))
        except Exception:
            pass
    nomi_set.update(aziende_processate)
    progress_path.write_text(
        json.dumps(sorted(nomi_set), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def load_progress(output_dir: Path) -> set[str]:
    """Carica la lista di aziende già processate."""
    progress_path = output_dir / "progresso.json"
    if not progress_path.exists():
        return set()
    try:
        return set(json.loads(progress_path.read_text(encoding="utf-8")))
    except Exception:
        return set()


print("Funzioni di salvataggio definite")


Funzioni di salvataggio definite


In [ ]:
# ▶ Salva i risultati
if risultati:
    log.info("--- Step 4: salvataggio output ---")
    tabella = save_output(risultati, OUTPUT, append=RESUME)
    save_progress([az["nome"] for az in batch], OUTPUT)
    print(f"\n💾 Output salvato in: {OUTPUT}/")
    display(tabella.head(10))
else:
    save_progress([az["nome"] for az in batch], OUTPUT)
    print("Nessun evento da salvare — progresso aggiornato")


09:59:09 [INFO] --- Step 4: salvataggio output ---
09:59:09 [INFO] Append: 20 nuovi + 145 precedenti = 165 totali
09:59:09 [INFO] Salvati: output/eventi_estratti.xlsx  e  output/eventi_estratti.json  (165 eventi totali)



💾 Output salvato in: output/


,azienda,ragione_sociale,codice_fiscale,data,tipo_evento,tipo_round,importo_eur,investitori_in,soci_uscenti,confidence,verifica,evidence,fonte_url
125,AMC Innovative,AMC INNOVATIVE SRL,2124330677,2025-09-09,funding_round,primary,8000000.0,,,0.9,verificato,We are thrilled that AlphaVest's shareholders ...,https://finance.yahoo.com/news/alphavest-acqui...
103,ATP Science Technologies,ATP SCIENCE TECHNOLOGIES SOCIETA' A RESPONSABI...,8470190722,2020-07,funding_round,primary,NaN,Cortina Capital,,0.9,verificato,BDO successfully managed the completion of the...,https://www.bdo.com.au/en-au/deals/nutritional...
158,Adorea,ADOREA S.R.L.,4114790134,NaN,funding_round,primary,3000000.0,DFF Ventures (Dutch Founders Fund); Silence; F...,,0.9,scartato,Adorea ha deliberato un aumento di capitale fi...,https://www.linkedin.com/posts/paolo-vitaloni-...
46,Agripasta,AGRIPASTA SRL SOCIETA' BENEFIT,4308100983,2024-03-15,funding_round,primary,NaN,Fondo Italiano d’Investimento,Webcor Investments Ltd,0.9,verificato,Fondo Italiano d’Investimento SGR annuncia il ...,https://www.fondoitaliano.it/fondo-italiano-di...
63,Algor LAB,ALGOR LAB S.R.L.,12537010014,2021-11,funding_round,primary,180000.0,Club degli investitori di Torino,,0.9,verificato,"Lo scorso novembre, Algor ha raccolto un inves...",https://www.humaneworldmagazine.com/algor-lab
59,Algor LAB,ALGOR LAB S.R.L.,12537010014,2021-11-17,funding_round,primary,180000.0,Club degli Investitori,,0.9,verificato,"Algor ha chiuso un round da 180 mila euro, gui...",https://www.simonfiduciaria.it/news/comunicati...
58,Algor LAB,ALGOR LAB S.R.L.,12537010014,2023-12-18,funding_round,primary,1400000.0,,,0.9,verificato,Its most recent round was a Seed of $1.5M in D...,https://fundediq.co/algor-lab-algoreducation-c...
61,Algor LAB,ALGOR LAB S.R.L.,12537010014,2023-12-18,funding_round,primary,1400000.0,Emerge Education; Club degli Investitori; 40Je...,,0.9,verificato,"Algor Education, startup innovativa che sta ri...",https://www.i3p.it/news/investimento-internazi...
64,Algor LAB,ALGOR LAB S.R.L.,12537010014,2023-12-18,funding_round,primary,1400000.0,Emerge Education; Club degli Investitori; 40Je...,,0.9,verificato,Algor Education announces that it has closed a...,https://www.startupbusiness.it/en/algor-raises...
60,Algor LAB,ALGOR LAB S.R.L.,12537010014,2024-01-03,funding_round,primary,1400000.0,Emerge Education; Club degli Investitori; 40Je...,,0.9,verificato,La startup edtech torinese Algor Education ric...,https://torinotechmap.it/notizie/startup-edtec...
